# Handwritten Digit Recognition with TensorFlow

In this notebook we'll build a simple **neural network** that learns to recognise handwritten digits (0-9).

We'll use the famous **MNIST dataset** — 70,000 small black-and-white images of handwritten digits, each 28x28 pixels.

### What we'll do, step by step:
1. Load the dataset
2. Look at the data so we understand what we're working with
3. Prepare ("preprocess") the data for training
4. Build a simple training model using TensorFlow/Keras
5. Train the model on the data
6. Test how well it performs, and understand what that result means
7. Use the model to make predictions and visualise the results
8. Test the model on your own handwriting, and even correct it live
9. Save the trained model, and see how we could make it even better

No prior machine learning experience is required — every step is explained with comments and plain-English explanations along the way, so treat this as a self-contained learning exercise rather than just a script to run.

## Step 1: Import the libraries we need

- **TensorFlow** is the machine learning library we'll use to build and train our model.
- **matplotlib** lets us draw pictures (so we can look at the digit images).
- **numpy** helps us work with arrays of numbers.

In [ ]:
# Import TensorFlow - the main library for building and training our model
import tensorflow as tf

# Import tools for showing images and working with numbers
import matplotlib.pyplot as plt
import numpy as np

# Print the TensorFlow version so we know what we're working with
print("TensorFlow version:", tf.__version__)

## Step 2: Load the MNIST dataset

TensorFlow comes with the MNIST dataset built in, so we don't need to download anything ourselves.

The data is split into two parts:
- **Training data**: used to teach the model
- **Testing data**: used afterwards to check how well the model learned (it never sees this data during training)

Each image comes with a **label** — the correct digit (0-9) that the image shows.

In [ ]:
# Load the MNIST dataset directly from Keras (part of TensorFlow)
mnist = tf.keras.datasets.mnist

# This gives us four things:
# x_train / y_train -> images and labels used for TRAINING the model
# x_test  / y_test  -> images and labels used for TESTING the model afterwards
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Let's check the shape (dimensions) of our data
print("Training images shape:", x_train.shape)  # e.g. (60000, 28, 28) -> 60,000 images of 28x28 pixels
print("Training labels shape:", y_train.shape)  # e.g. (60000,)        -> 60,000 labels
print("Testing images shape:", x_test.shape)
print("Testing labels shape:", y_test.shape)

## Step 3: View 20 random handwritten digits

Now that we've loaded the MNIST dataset, let's look at the data — 20 randomly chosen handwritten digits, arranged in a 4x5 grid, each with its correct label underneath. Run the cell again to see a different random selection each time.

In [ ]:
# Pick 20 random indices from the training set, so we see a different batch each run
random_indices = np.random.choice(len(x_train), size=20, replace=False)

# Create a figure with a 4x5 grid, so we can show 20 images at once
plt.figure(figsize=(10, 8))

for plot_position, index in enumerate(random_indices):
    plt.subplot(4, 5, plot_position + 1)     # 4 rows, 5 columns, position plot_position+1
    plt.imshow(x_train[index], cmap="gray")  # Show the image in grayscale
    plt.title(f"Label: {y_train[index]}")    # Show the correct digit above the image
    plt.axis("off")                          # Hide the x/y axis numbers, they're not needed

plt.tight_layout()
plt.show()

## Step 4: Prepare the data (preprocessing)

Each pixel in an image is a number from **0 to 255** (0 = black, 255 = white).

Neural networks train better when numbers are small, so we'll **scale** every pixel value down to a range of **0 to 1** by dividing by 255. This is called *normalisation*.

In [ ]:
# Scale pixel values from the range [0, 255] down to [0, 1]
# This helps the model train faster and more reliably
x_train = x_train / 255.0
x_test = x_test / 255.0

print("Smallest pixel value:", x_train.min())
print("Largest pixel value:", x_train.max())

## Step 5: Build the training model

We'll use a **Sequential model** — the simplest kind of neural network in Keras, where layers are stacked one after another, each feeding its output into the next.

Our model has 4 layers:
1. **Flatten**: turns each 28x28 image into a single flat list of 784 numbers (28 x 28 = 784). Neural networks like this one expect a simple list of numbers as input, not a 2D grid.
2. **Dense (128 neurons)**: a "fully-connected" layer — every one of its 128 neurons looks at all 784 input numbers and learns to combine them in a way that's useful for recognising digits. It uses the **"relu" activation function**, which simply turns any negative number into 0 and leaves positive numbers unchanged. This small non-linear "kink" is what lets the network learn complex, non-straight-line patterns rather than just a simple weighted average.
3. **Dropout**: during training only, this layer randomly "switches off" 20% of the neurons from the previous layer on each pass. This stops the network from relying too heavily on any single neuron, which helps prevent **overfitting** (memorising the training images instead of learning general patterns).
4. **Dense (10 neurons)**: the output layer — one neuron for each possible digit (0-9). It uses the **"softmax" activation function**, which turns the 10 raw output numbers into probabilities that all add up to 1 (100%) — e.g. "80% chance this is a 3, 15% chance it's an 8, 5% spread across everything else".

### What to expect

When you run the cell below, `model.summary()` prints a table describing each layer. Here's how to read it:

| Column | What it means |
|---|---|
| **Layer (type)** | The kind of layer and Keras' auto-generated name for it |
| **Output Shape** | The shape of data coming *out* of that layer. `(None, 128)` means "any number of images, each represented by 128 numbers" — the `None` is a placeholder for batch size, since we can feed in 1 image or 1,000 at once |
| **Param #** | The number of learnable numbers (weights and biases) in that layer |

You should see a **Total params** count of **101,770** — all of them trainable, since we haven't frozen any part of the model. That number comes from: (784 inputs x 128 neurons + 128 biases) in the first Dense layer, plus (128 x 10 + 10 biases) in the output layer. Flatten and Dropout have no parameters of their own — they don't *learn* anything, they just reshape or randomly disable data as it passes through.

In [ ]:
# Build a simple Sequential (stacked-layers) model
model = tf.keras.models.Sequential([

    # Tell the model what shape of data to expect: 28x28 pixel images
    tf.keras.layers.Input(shape=(28, 28)),

    # Turn each 28x28 image into a flat list of 784 numbers
    tf.keras.layers.Flatten(),

    # A layer of 128 neurons that learns patterns from the pixels
    tf.keras.layers.Dense(128, activation="relu"),

    # Randomly ignore 20% of neurons during training to reduce overfitting
    tf.keras.layers.Dropout(0.2),

    # Output layer: 10 neurons, one for each digit (0-9)
    # "softmax" turns the output into probabilities that add up to 1
    tf.keras.layers.Dense(10, activation="softmax")
])

# Print a summary of the model's layers
model.summary()

## Step 6: Compile the model

"Compiling" doesn't train the model yet — it just configures *how* training will work, by defining three things:

- **Optimizer** (`"adam"`): the algorithm that decides how to adjust the model's weights after each look at the data, so it gets a little better each time. Adam is a widely-used, reliable choice because it automatically adapts how big a step to take — bigger steps when it's far from a good answer, smaller and more careful steps as it gets close. You can think of it as trial-and-error, but a very smart, structured version of it.
- **Loss function** (`"sparse_categorical_crossentropy"`): a formula that scores how *wrong* a prediction was, as a single number. A perfect prediction (100% confidence in the correct digit) scores a loss near 0; a confidently wrong prediction scores a high loss. During training, the optimizer's whole job is to adjust the weights to make this number smaller. We use the "sparse" version because our labels are plain integers (like `7`), rather than one-hot encoded vectors (like `[0,0,0,0,0,0,0,1,0,0]`) — "categorical crossentropy" is the standard loss function for classifying into multiple categories.
- **Metrics** (`"accuracy"`): unlike the loss, metrics aren't used to guide training — they're purely for *us* to monitor progress in a way that's easier to interpret than a loss number. Accuracy simply means "what percentage of predictions were exactly correct".

In [ ]:
model.compile(
    optimizer="adam",                             # How the model improves itself
    loss="sparse_categorical_crossentropy",       # How we measure prediction error for multiple classes
    metrics=["accuracy"]                          # Track accuracy while training
)

## Step 7: Train the model

Now we let the model learn from the training data. This is the step that actually adjusts the model's weights, using the optimizer and loss function we set up in Step 6.

Here's roughly what happens on repeat, thousands of times, during training:
1. The model looks at a small "batch" of training images (32 at a time, by default) and makes predictions
2. The loss function measures how wrong those predictions were
3. The optimizer works out how to nudge every weight in the model to make that loss slightly smaller
4. The weights are updated, and the process repeats with the next batch

An **epoch** means one full pass through all 60,000 training images. We'll train for 5 epochs — enough to get good accuracy quickly on a dataset this size. Each epoch, you'll see a progress bar followed by two numbers:

- **loss**: should get *smaller* each epoch — a lower score means the model's predictions are getting closer to correct
- **accuracy**: should get *larger* each epoch — this is the percentage of training images it's currently getting right

Training runs entirely on your CPU here, and for a small dataset and model like this one, it should take well under a minute in total (a few seconds per epoch). The cell won't print anything until an epoch finishes, so it's normal to see no output for a moment after you run it.

**Note:** the cell below will show `[*]` next to it (in Jupyter) while it's still running — wait for that to change to a number before moving on to the next cell, otherwise you'll be evaluating a model that hasn't finished training yet.

In [ ]:
# Let the user know this step takes a little while, so they don't move on too early
print("Training the model - please wait for this to finish before running the next cell...\n")

# Train the model on the training data for 5 epochs
history = model.fit(x_train, y_train, epochs=5)

print("\nTraining complete! You can now move on to the next cell.")

## Step 8: Evaluate the model

Now let's see how well the model performs on the **test data** — 10,000 images it has never seen during training. This is an important, honest check: since the model never learned from these specific images, its performance here tells us how well it **generalises** to new, unseen digits — which is really what we care about.

### Understanding the test accuracy

With this setup, you should see a test accuracy somewhere around **97-98%**. That means for every 100 digits it's shown, it correctly identifies roughly 97 or 98 of them.

You may also notice the test accuracy is very close to (sometimes even slightly *lower* than) the training accuracy from the last epoch of Step 7. That's expected and healthy — it means the model learned general patterns rather than memorising the training images. If test accuracy were *much* lower than training accuracy (e.g. 99% training vs 80% test), that would be a sign of **overfitting**: the model memorised quirks of the training images instead of learning what actually makes a "7" look like a "7".

### If your accuracy seems low

A few common causes, and how to fix them:

- **Too few epochs**: 5 epochs is enough for a good result here, but if you experiment and reduce it, accuracy will drop. Try training for longer (Step 7).
- **Data not normalised**: if the pixel-scaling step (Step 4) was skipped or run twice, the model receives numbers in the wrong range and struggles to learn efficiently.
- **A model that's too simple**: our single hidden layer of 128 neurons works well for MNIST, but for harder image problems you'd need a more powerful architecture — see the Convolutional Neural Network (CNN) note at the end of this notebook.
- **Randomness**: neural networks start with random initial weights and Dropout randomly disables neurons during training, so re-running training from scratch can give a slightly different result each time (typically within a percent or so) — this is normal, not a bug.
- **Overfitting**: if you add many more layers/neurons without enough Dropout, the model can start memorising the training set. Watch for training accuracy pulling far ahead of test accuracy as a warning sign.

In [ ]:
# Check the model's performance on the unseen test data
test_loss, test_accuracy = model.evaluate(x_test, y_test)

print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")

## Step 9: Make predictions

Let's use our trained model to predict some digits from the test set, and compare them to the real answers.

### What are we actually feeding the model?

`x_test[0]` is a single 28x28 grid of numbers between 0 and 1 (we normalised it back in Step 4) — one number per pixel, where 0 is black and 1 is white. It's simply the first image in the test set; you saw digits just like it back in Step 3. There's nothing special about "index 0" — it's just the first one in the array, and `y_test[0]` holds its true label.

### How does the model turn that into an answer?

When we call `model.predict(x_test)`, each image flows through the layers we built in Step 5 (Flatten → Dense → Dropout → Dense), and the final softmax layer outputs **10 numbers between 0 and 1 that add up to 1** — one probability per digit. For example, an output of:

```
[0.01, 0.00, 0.02, 0.01, 0.00, 0.00, 0.00, 0.94, 0.00, 0.02]
```

means the model is 94% confident the image is a **7** (the digit at index 7), with small amounts of leftover uncertainty spread across a few other digits it thinks are less likely. `np.argmax(...)` simply finds the *position* of the largest number in that list — in this example, position 7 — which becomes the model's final answer.

In [ ]:
# Ask the model to predict probabilities for every test image
predictions = model.predict(x_test)

# For the very first test image, show the probability for each digit
print("Predicted probabilities for the first test image:")
print(np.round(predictions[0], 2))

# The digit with the highest probability is the model's final answer
predicted_digit = np.argmax(predictions[0])
print("Predicted digit:", predicted_digit)
print("Actual digit:", y_test[0])

### Seeing the full probability array, for several examples

Numbers in a list are hard to build intuition from, so let's plot the probability array as a bar chart, next to the image that produced it, for a few different random examples. The tallest bar is always the model's final answer — but watching *how tall* it is, and whether any other bars come close, shows you the model's confidence, not just its answer.

In [ ]:
# Pick a few random test images to inspect closely
example_indices = np.random.choice(len(x_test), size=4, replace=False)

plt.figure(figsize=(12, 6))

for row, index in enumerate(example_indices):
    # Left column: the actual image
    plt.subplot(4, 2, row * 2 + 1)
    plt.imshow(x_test[index], cmap="gray")
    predicted_digit = np.argmax(predictions[index])
    color = "green" if predicted_digit == y_test[index] else "red"
    plt.title(f"True: {y_test[index]}  |  Predicted: {predicted_digit}", color=color)
    plt.axis("off")

    # Right column: a bar chart of all 10 probabilities for this image
    plt.subplot(4, 2, row * 2 + 2)
    bar_colors = ["green" if digit == y_test[index] else "steelblue" for digit in range(10)]
    plt.bar(range(10), predictions[index], color=bar_colors)
    plt.xticks(range(10))
    plt.ylim(0, 1)
    plt.ylabel("Probability")
    plt.xlabel("Digit")

plt.tight_layout()
plt.show()

## Step 10: Visualise predictions

Let's look at several test images side by side with the model's predicted digit and the true digit.
Correct predictions are shown in green, wrong ones in red.

In [ ]:
# Show the first 10 test images with their predicted vs actual labels
plt.figure(figsize=(12, 5))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i], cmap="gray")
    plt.axis("off")

    predicted_digit = np.argmax(predictions[i])
    actual_digit = y_test[i]

    # Green title if correct, red if wrong
    color = "green" if predicted_digit == actual_digit else "red"
    plt.title(f"Pred: {predicted_digit}\nTrue: {actual_digit}", color=color)

plt.tight_layout()
plt.show()

## Step 11: Draw your own digit and test the model!

The real test of our model is whether it can recognise *your* handwriting, not just MNIST's.

Below is a small drawing canvas. Draw a single digit (0-9) with your mouse, then click **Predict** to see what the model thinks it is. Click **Clear** to try again.

Once you have a prediction, you'll be asked whether it was correct:
- If it was, great — nothing more to do.
- If it wasn't, you can tell the model the correct digit and click **Teach the model**. This runs one quick training step on your drawing, nudging the model's weights so it's more likely to get that digit right next time. This is a simple example of *online learning* — updating a model from new examples after it's already been trained.

A few tips for best results:
- Draw a large digit that fills most of the canvas (MNIST digits are big and centred)
- Use a single, fairly thick stroke
- The model was only ever trained on MNIST's style of digit — don't be surprised if it gets confused by unusual handwriting!

### Step 11a: Set up the drawing canvas

In [ ]:
# ipycanvas gives us a drawable canvas widget; ipywidgets gives us buttons and layout
from ipycanvas import Canvas
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

# The canvas is 280x280 (10x the size of a 28x28 MNIST image) so it's easy to draw on
CANVAS_SIZE = 280

# sync_image_data=True lets us read back what was drawn, from Python
canvas = Canvas(width=CANVAS_SIZE, height=CANVAS_SIZE, sync_image_data=True)

# MNIST digits are white strokes on a black background, so we draw the same way
canvas.fill_style = "black"
canvas.fill_rect(0, 0, CANVAS_SIZE, CANVAS_SIZE)
canvas.stroke_style = "white"
canvas.line_width = 20
canvas.line_cap = "round"

# Keep track of whether the mouse button is currently held down
is_drawing = False

def start_drawing(x, y):
    global is_drawing
    is_drawing = True
    canvas.begin_path()
    canvas.move_to(x, y)

def draw_line(x, y):
    if is_drawing:
        canvas.line_to(x, y)
        canvas.stroke()

def stop_drawing(x, y):
    global is_drawing
    is_drawing = False

canvas.on_mouse_down(start_drawing)
canvas.on_mouse_move(draw_line)
canvas.on_mouse_up(stop_drawing)

### Step 11b: Predict, confirm, and learn from corrections

In [ ]:
def preprocess_drawing(canvas):
    """Turn what's drawn on the canvas into a 28x28 image the model can understand."""

    # Read the canvas as an array of pixels (each pixel has Red, Green, Blue, Alpha)
    image_data = canvas.get_image_data()

    # Our drawing is white-on-black, so the Red channel alone tells us brightness (0-255)
    brightness = image_data[:, :, 0]

    # Find the box that tightly contains everything we've drawn
    drawn_rows = np.any(brightness > 0, axis=1)
    drawn_cols = np.any(brightness > 0, axis=0)

    if not drawn_rows.any():
        return None  # Nothing has been drawn yet

    top, bottom = np.where(drawn_rows)[0][[0, -1]]
    left, right = np.where(drawn_cols)[0][[0, -1]]

    # Crop to the drawn digit, then add a border so it isn't touching the edges
    # (MNIST digits are centred with some empty space around them)
    digit = Image.fromarray(brightness[top:bottom + 1, left:right + 1].astype("uint8"))
    digit.thumbnail((20, 20))  # Shrink to fit in a 20x20 box, keeping its proportions

    canvas_28 = Image.new("L", (28, 28), color=0)  # Blank 28x28 black image
    paste_x = (28 - digit.width) // 2
    paste_y = (28 - digit.height) // 2
    canvas_28.paste(digit, (paste_x, paste_y))  # Centre the digit in the middle

    # Scale pixel values to 0-1, just like we did with the training data
    return np.array(canvas_28) / 255.0


# These remember the most recent prediction, so the feedback buttons below can use them
last_drawn_image = None
last_predicted_digit = None

result_output = widgets.Output()    # Shows the drawn digit and the model's prediction
feedback_output = widgets.Output()  # Shows the "was this correct?" controls


def show_feedback_prompt():
    """Ask the user whether the last prediction was correct."""
    feedback_output.clear_output()
    with feedback_output:
        print("Was that correct?")
        display(widgets.HBox([correct_button, incorrect_button]))


def on_predict_clicked(button):
    global last_drawn_image, last_predicted_digit

    result_output.clear_output(wait=True)
    with result_output:
        processed_image = preprocess_drawing(canvas)
        if processed_image is None:
            print("Draw a digit first!")
            feedback_output.clear_output()
            return

        # The model expects a batch of images, so we add an extra dimension
        prediction = model.predict(processed_image.reshape(1, 28, 28), verbose=0)
        predicted_digit = np.argmax(prediction)
        confidence = np.max(prediction) * 100

        # Remember this prediction so the feedback buttons know what they're correcting
        last_drawn_image = processed_image
        last_predicted_digit = predicted_digit

        # Show exactly what the model saw, next to its answer
        plt.figure(figsize=(3, 3))
        plt.imshow(processed_image, cmap="gray")
        plt.title(f"Predicted: {predicted_digit} ({confidence:.1f}% confident)")
        plt.axis("off")
        plt.show()

    show_feedback_prompt()


def on_clear_clicked(button):
    global last_drawn_image, last_predicted_digit
    canvas.fill_style = "black"
    canvas.fill_rect(0, 0, CANVAS_SIZE, CANVAS_SIZE)
    last_drawn_image = None
    last_predicted_digit = None
    result_output.clear_output()
    feedback_output.clear_output()


def on_correct_clicked(button):
    feedback_output.clear_output()
    with feedback_output:
        print("Great! Glad it got that one right.")


def on_incorrect_clicked(button):
    # Let the user pick the real digit, then teach the model with it
    digit_picker = widgets.Dropdown(options=list(range(10)), description="Actual digit:")
    teach_button = widgets.Button(description="Teach the model", button_style="info")

    def on_teach_clicked(button):
        # A single training step on this one example, using the correct label.
        # This is a toy version of "online learning" - nudging the model's weights
        # with one new example, rather than retraining from scratch. A few repeats
        # (epochs) help the correction sink in, but doing this too often on too few
        # examples can make the model overfit to your handwriting and forget general
        # patterns it learned from MNIST - so treat it as a fun demo, not a real
        # training strategy.
        correct_label = np.array([digit_picker.value])
        model.fit(last_drawn_image.reshape(1, 28, 28), correct_label, epochs=3, verbose=0)

        feedback_output.clear_output()
        with feedback_output:
            print(f"Thanks! The model just learned this drawing is a {digit_picker.value}.")

    teach_button.on_click(on_teach_clicked)

    feedback_output.clear_output()
    with feedback_output:
        print("Sorry about that! What digit did you actually draw?")
        display(widgets.HBox([digit_picker, teach_button]))


correct_button = widgets.Button(description="Yes, correct", button_style="success")
incorrect_button = widgets.Button(description="No, wrong", button_style="danger")
correct_button.on_click(on_correct_clicked)
incorrect_button.on_click(on_incorrect_clicked)

predict_button = widgets.Button(description="Predict", button_style="success")
clear_button = widgets.Button(description="Clear", button_style="warning")
predict_button.on_click(on_predict_clicked)
clear_button.on_click(on_clear_clicked)

# Show the canvas, the buttons, the prediction, and the feedback controls together
display(canvas, widgets.HBox([predict_button, clear_button]), result_output, feedback_output)

## Step 12: Save the trained model (optional)

Once we're happy with our model, we can save it to a file so we can reuse it later without retraining.

### What is a `.keras` file?

Training this model cost us time and computing effort — the `.keras` file lets us keep the *result* of that work. It's a zip-style archive that bundles together everything needed to use (or continue training) the model later, without starting from scratch:

- **The architecture**: the layers we defined in Step 5 and how they connect
- **The learned weights**: every number the model adjusted during training in Step 7 — this is the actual "knowledge" the model gained
- **The optimizer's state**: so you could even resume training later exactly where you left off
- **The compile settings** from Step 6 (loss function, metrics, etc.)

`.keras` is the modern, recommended format for Keras 3 (which is what we're using here). You may also see the older `.h5` format in other tutorials — it stores similar information but is being phased out in favour of `.keras`. Either way, a saved model file is just data (numbers describing the network) — running it still requires TensorFlow/Keras installed to load and use it, the same way a saved Word document needs Word (or a compatible program) installed to open it.

In [ ]:
# Save the trained model to a file
model.save("handwritten_digit_model.keras")
print("Model saved as handwritten_digit_model.keras")

# To load it again later, you would use:
# model = tf.keras.models.load_model("handwritten_digit_model.keras")

## Summary

This notebook walked through the complete, standard workflow for a supervised machine learning
project — the same broad steps apply whether you're recognising digits, classifying emails as
spam, or predicting house prices:

1. **Get labelled data** — we loaded MNIST, which comes with 70,000 images already labelled with their correct digit (Step 2)
2. **Explore it** — looking at real examples before doing anything else helps you understand what you're working with (Step 3)
3. **Prepare it** — we normalised pixel values to a 0-1 range, which helps the model train efficiently (Step 4)
4. **Design a model** — we chose a simple architecture: layers that flatten, learn patterns, and randomly drop out to avoid overfitting (Step 5)
5. **Configure training** — compiling set up *how* the model would learn: an optimizer, a loss function, and a metric to watch (Step 6)
6. **Train** — the model repeatedly looked at data, measured its mistakes, and adjusted itself to make fewer of them (Step 7)
7. **Evaluate honestly** — checking accuracy on data the model never saw during training tells us how well it actually generalises (Step 8)
8. **Use the model** — generating predictions and inspecting the probabilities behind them (Step 9-10)
9. **Test it in the real world** — your own handwriting is very different from a curated dataset, and the drawing canvas let you feel that gap directly, along with a taste of *online learning* by correcting the model live (Step 11)
10. **Persist the result** — saving the trained model so the work isn't lost (Step 12)

### Key terms recap

| Term | What it means |
|---|---|
| **Neural network / model** | A system of connected layers that learns to map inputs (images) to outputs (digit labels) by adjusting internal numbers called weights |
| **Training** | The process of adjusting a model's weights using example data, to reduce its mistakes |
| **Epoch** | One full pass through the entire training dataset |
| **Loss** | A single number scoring how wrong the model's predictions were; training tries to minimise this |
| **Accuracy** | The percentage of predictions that were exactly correct — easier for humans to interpret than loss |
| **Overfitting** | When a model memorises quirks of the training data instead of learning general patterns, so it does much worse on new data |
| **Normalisation** | Rescaling input data (here, pixel values) into a small, consistent range to help training work well |

### Ideas to try next

- Train for more epochs and see if accuracy improves (and watch for diminishing returns, or even overfitting, if you push this too far)
- Add more Dense layers, or change the number of neurons, and see how that affects both accuracy and training time
- See how the model handles messy or unusually-shaped handwriting on the drawing canvas, and use the "teach the model" feature to correct it live
- Curious what's actually happening inside `model.fit()`? See the **Bonus: Watch backpropagation happen** section below, which computes real gradients by hand using `tf.GradientTape()`
- Try a **Convolutional Neural Network (CNN)** instead — usually noticeably better at image recognition. Here's what that means:

#### What is a CNN, and how is it different from what we built?

Our model starts by *flattening* each image into a flat list of 784 numbers (Step 5). That works reasonably well for MNIST, but it throws away something important: which pixels were *next to* each other. A "7" is defined by pixels forming particular shapes and edges in particular spatial arrangements — flattening loses that spatial structure entirely, treating pixel 1 and pixel 785 as no more or less related than any other pair.

A **Convolutional Neural Network** keeps the image in its 2D grid shape and uses a different kind of layer — a **convolutional layer** — that works more like a sliding magnifying glass than a flat list-reader:

- It scans a small **filter** (commonly 3x3 pixels) across the whole image, a few pixels at a time
- At each position, that filter checks for one specific small pattern — like a vertical edge, a curve, or a corner
- The result is a new grid (called a **feature map**) showing *where* in the image that pattern was found
- A CNN stacks several of these layers: early layers detect simple things like edges and curves, and later layers combine those into more complex shapes (loops, corners, strokes) — much like how the digit "8" is really just two loops stacked together

This gives CNNs two big advantages for images:
1. **Parameter sharing**: the same small filter is reused across the entire image, rather than needing a separate weight for every single pixel combination — this means far fewer parameters to learn for a given amount of "pattern-spotting power"
2. **Translation invariance**: because the filter slides across the whole image, a CNN can recognise a pattern (like a curve) wherever it appears in the picture, not just in one fixed spot — helpful since nobody draws digits in exactly the same position twice

A simple CNN for MNIST typically replaces the first Flatten/Dense pair from Step 5 with a couple of `Conv2D` and `MaxPooling2D` layers (which shrink the feature maps down, keeping only the strongest signals) before finally flattening near the end. It usually takes a bit longer to train, but often reaches 99%+ test accuracy on MNIST, compared to our ~97-98%.

Curious to actually try one? See the **Bonus** section below — it shows exactly which cells above you'd need to change, and includes a runnable CNN you can compare directly against the model we just built.

## Bonus: Watch backpropagation happen

Every time you called `model.fit(...)` in Step 7, **backpropagation** ran thousands of times
behind the scenes — once for every batch, every epoch. It's the algorithm that makes training
possible: it works out exactly how much each individual weight in the network contributed to the
loss, so the optimizer knows which direction (and how far) to nudge it.

Keras hides this completely inside `.fit()`. This bonus section un-hides it, using
`tf.GradientTape()` — TensorFlow's tool for recording every calculation as it happens, so it can
be "played backwards" afterwards to work out gradients. That recording-then-replaying-backwards
is, quite literally, what "backpropagation" means.

### The core idea, before any neural network is involved

At its heart, backpropagation is just the **chain rule** from calculus, applied automatically.
Let's see it on the simplest possible example first: one input number, one weight, one output —
no layers, no images, nothing hidden. This is small enough to check TensorFlow's answer by hand.

In [ ]:
# The simplest possible "model": one weight, predicting prediction = w * x
x = tf.constant(3.0)         # a fixed input
target = tf.constant(10.0)   # what we wish the prediction was
w = tf.Variable(2.0)         # our one and only "weight" - tf.Variable so TF tracks changes to it

# tf.GradientTape() records every calculation performed inside it
with tf.GradientTape() as tape:
    prediction = w * x                    # forward pass
    loss = (prediction - target) ** 2     # squared error: how wrong were we?

# This is the backward pass: tape.gradient() replays everything recorded above,
# backwards, applying the chain rule to work out d(loss)/dw - how much a tiny
# nudge to w would change the loss
gradient = tape.gradient(loss, w)

print(f"prediction = w * x = {w.numpy()} * {x.numpy()} = {prediction.numpy()}")
print(f"loss = (prediction - target)^2 = ({prediction.numpy()} - {target.numpy()})^2 = {loss.numpy()}")
print(f"\nGradient TensorFlow calculated: {gradient.numpy()}")

# Let's check that by hand, using the chain rule ourselves:
# loss = (w*x - target)^2
# d(loss)/dw = 2 * (w*x - target) * x
manual_gradient = 2 * (w.numpy() * x.numpy() - target.numpy()) * x.numpy()
print(f"Gradient calculated by hand:    {manual_gradient}")

Both numbers should match exactly. TensorFlow didn't guess, look anything up, or approximate —
it mechanically applied the same chain rule you'd use with pen and paper, just automatically.

### Now let's do the same thing on our real, trained model

Our model has 101,770 weights instead of 1, and the calculations chain through four layers
instead of one multiplication — but the *mechanism* is identical: record the forward pass, then
ask for gradients. Let's run it on a real batch of training images and look at some actual
gradient values.

In [ ]:
# Grab one real batch from the training data
sample_images = x_train[:32]
sample_labels = y_train[:32]

with tf.GradientTape() as tape:
    predictions_sample = model(sample_images, training=True)   # forward pass
    loss_value = tf.keras.losses.sparse_categorical_crossentropy(sample_labels, predictions_sample)
    loss_value = tf.reduce_mean(loss_value)

print(f"Loss for this batch: {loss_value.numpy():.4f}\n")

# The backward pass: walk backwards through every layer used above, applying the
# chain rule at each one, until we have a gradient for every single trainable weight
gradients = tape.gradient(loss_value, model.trainable_variables)

# Look closely at the gradients for just the first layer's weights
first_layer_weights = model.trainable_variables[0]
first_layer_gradients = gradients[0]

print(f"First layer weight matrix shape: {first_layer_weights.shape}")
print(f"Matching gradient shape:         {first_layer_gradients.shape}")

# Pixel 400 sits roughly in the middle of the 28x28 image, where digit strokes
# actually appear - a more informative example than a corner pixel (see note below)
print(f"\nA few example gradient values (from pixel index 400):\n{first_layer_gradients.numpy()[400, :5]}")

### Reading the gradients

The gradient matrix has exactly the same shape as the weight matrix it belongs to — one gradient
value for every single weight, telling that specific weight two things:

- **Sign**: whether *increasing* that weight would increase or decrease the loss
- **Magnitude**: how much difference that weight makes right now — a value near 0 means "barely
  matters at the moment", a larger value means "this one has a real effect on the loss"

**Try it yourself:** change the pixel index above from `400` to `0` and re-run the cell. You'll
get back exactly `[0. 0. 0. 0. 0.]` — not "very small", but *exactly* zero. That's not a bug: pixel
0 is a corner of the image, and MNIST digits are always centred with black borders, so that pixel
is 0 in essentially every training image. A weight that only ever multiplies against 0 can never
affect the loss, so its gradient is always exactly 0 too — the network correctly has nothing to
learn about that particular pixel.

The optimizer (Step 6) uses exactly this information, for all 101,770 weights at once, to decide
how to adjust each one. Let's prove it by applying one real update and watching a weight change.

In [ ]:
# Save the model's current weights, so we can restore them afterwards - this is just a
# demonstration, and we don't want it to affect the model used in the rest of the notebook
original_weights = model.get_weights()

# Use the same pixel-400 row as above, which we know has a real (non-zero) gradient
weight_before = first_layer_weights.numpy()[400, 0]

# Apply one real optimizer step using the gradients we just calculated - this is the
# exact same operation that happens inside model.fit(), just done here by hand
model.optimizer.apply_gradients(zip(gradients, model.trainable_variables))

weight_after = model.trainable_variables[0].numpy()[400, 0]

print(f"Weight value before this update: {weight_before:.8f}")
print(f"Weight value after this update:  {weight_after:.8f}")
print(f"Change:                          {weight_after - weight_before:.8f}")

# Restore the original weights, undoing our one manual update
model.set_weights(original_weights)
print("\n(Model weights restored to their trained state - this was just a demonstration.)")

## Bonus: Try a CNN yourself

### Where in the code above you'd need to change things

If you wanted to convert the *whole* notebook above to use a CNN instead of our simple Dense
network, here's exactly what would need to change, cell by cell:

- **Step 4 (Prepare the data)**: our Dense-based model reads each image as a flat 784-number
  list, so `x_train`/`x_test` are 3D arrays shaped `(num_images, 28, 28)`. Conv2D layers instead
  expect a 4D array shaped `(num_images, 28, 28, channels)` — an extra dimension for colour
  channels (1 for grayscale, like ours; 3 for RGB). You'd add:
  ```python
  x_train_cnn = x_train.reshape(-1, 28, 28, 1)
  x_test_cnn = x_test.reshape(-1, 28, 28, 1)
  ```

- **Step 5 (Build the model)**: swap the `Input` shape and the first `Flatten`/`Dense` pair for
  `Conv2D`/`MaxPooling2D` layers, keeping a `Flatten` only right before the final Dense layers:
  ```python
  model = tf.keras.models.Sequential([
      tf.keras.layers.Input(shape=(28, 28, 1)),          # note the extra "1" here
      tf.keras.layers.Conv2D(32, kernel_size=3, activation="relu"),
      tf.keras.layers.MaxPooling2D(pool_size=2),
      tf.keras.layers.Conv2D(64, kernel_size=3, activation="relu"),
      tf.keras.layers.MaxPooling2D(pool_size=2),
      tf.keras.layers.Flatten(),
      tf.keras.layers.Dense(64, activation="relu"),
      tf.keras.layers.Dropout(0.3),
      tf.keras.layers.Dense(10, activation="softmax")
  ])
  ```

- **Step 6 (Compile)**: no changes needed — the optimizer, loss, and metrics stay the same.

- **Step 7 (Train)**: use the reshaped data: `model.fit(x_train_cnn, y_train, epochs=5)`

- **Step 8 (Evaluate)**: use the reshaped data: `model.evaluate(x_test_cnn, y_test)`

- **Step 9 (Predict)**: use the reshaped data: `model.predict(x_test_cnn)`

- **Step 11b (Drawing canvas)**: the line `processed_image.reshape(1, 28, 28)` would become
  `processed_image.reshape(1, 28, 28, 1)`, for the same reason as Step 4 — the model now expects
  that extra channel dimension on every image it's given, including your own drawings.

- **Step 12 (Save)**: no changes needed — `model.save(...)` works the same regardless of architecture.

### Try it here, without touching anything above

Rather than editing the notebook above, the cell below builds and trains a small CNN
independently (using its own `_cnn` variable names), so you can compare it against our original
model without affecting anything else. It follows the same shape as the code above — feel free
to tweak the number of filters, layers, or epochs and re-run it.

**Note:** CNNs do more computation per image than our simple Dense network, so this cell will
generally take longer to train than Step 7 did — exactly how much longer depends on your CPU,
typically anywhere from several seconds to a couple of minutes. That trade-off (slower training,
usually better accuracy) is exactly the kind of decision you'll weigh often in machine learning.

In [ ]:
# Reshape the data to add the "channels" dimension Conv2D layers expect (see Step 4 above)
x_train_cnn = x_train.reshape(-1, 28, 28, 1)
x_test_cnn = x_test.reshape(-1, 28, 28, 1)

# A small CNN: two Conv2D + MaxPooling2D blocks to learn spatial patterns,
# then Dense layers (like our original model) to turn those patterns into a digit guess
cnn_model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, kernel_size=3, activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=2),
    tf.keras.layers.Conv2D(64, kernel_size=3, activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(10, activation="softmax")
])

cnn_model.summary()

# Same compile settings as our original model (Step 6) - only the architecture has changed
cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nTraining the CNN - this takes longer than Step 7, please wait...\n")
cnn_model.fit(x_train_cnn, y_train, epochs=3)
print("\nTraining complete!")

# Compare directly against our original Dense model's test accuracy from Step 8
cnn_test_loss, cnn_test_accuracy = cnn_model.evaluate(x_test_cnn, y_test)
print(f"\nCNN test accuracy:  {cnn_test_accuracy:.4f} ({cnn_test_accuracy * 100:.2f}%)")
print(f"Dense test accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)  (from Step 8)")